## 1. 데이터셋 조인
- [기준 데이터셋] KCGS_(2025~2025)_평가등급(2026).xlsx
- [보조] KOSDAQ업종분류현황.csv
- [보조] KOSPI업종분류현황.csv

-> 기준데이터셋에 보조데이터셋 2개를 조인해 기업명, 종목코드가 같은 것의 업종명, 종가, 대비, 등락률, 시가총액 컬럼을 추가함.

In [102]:
# ============================================
# 셀 1: 라이브러리 임포트
# ============================================
import pandas as pd
import os

print("✓ 라이브러리 로드 완료")

✓ 라이브러리 로드 완료


In [103]:
score_file = r'/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/전체/99_이전데이터셋/KCGS_(2025~2025)_평가등급(2026).xlsx'
kosdaq_file = r'/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/전체/99_이전데이터셋/KOSDAQ업종분류현황.csv'
kospi_file = r'/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/전체/99_이전데이터셋/KOSPI업종분류현황.csv'
output_file = r'/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_조인.csv'

print("✓ 파일 경로 설정 완료")

✓ 파일 경로 설정 완료


In [104]:
# ============================================
# 셀 3: 파일 읽기 (기업코드를 문자열로 읽기!)
# ============================================
# ⚠️ dtype={'기업코드': str} 으로 문자열로 읽음
df_excel = pd.read_excel(score_file, dtype={'기업코드': str})
 
# CSV 읽기 (한글 인코딩)
df_kosdaq = pd.read_csv(kosdaq_file, encoding='euc-kr', dtype={'종목코드': str})
df_kospi = pd.read_csv(kospi_file, encoding='euc-kr', dtype={'종목코드': str})
 
print("✓ 파일 읽기 완료 (기업코드/종목코드를 문자열로 로드)")
print(f"  - Excel: {len(df_excel)} 행")
print(f"  - KOSDAQ: {len(df_kosdaq)} 행")
print(f"  - KOSPI: {len(df_kospi)} 행")

✓ 파일 읽기 완료 (기업코드/종목코드를 문자열로 로드)
  - Excel: 1091 행
  - KOSDAQ: 1830 행
  - KOSPI: 958 행


In [105]:
# ============================================
# 셀 4: 데이터 구조 확인
# ============================================
print("\n" + "=" * 60)
print("Excel 기업코드 샘플:")
print(df_excel[['기업명', '기업코드']].head(5))
print(f"기업코드 데이터 타입: {df_excel['기업코드'].dtype}")
 
print("\n" + "=" * 60)
print("KOSDAQ 종목코드 샘플:")
print(df_kosdaq[['종목명', '종목코드', '시장구분']].head(5))
print(f"종목코드 데이터 타입: {df_kosdaq['종목코드'].dtype}")
 



Excel 기업코드 샘플:
          기업명    기업코드
0      AJ네트웍스  095570
1       AK홀딩스  006840
2         BGF  027410
3      BGF리테일  282330
4  BGF에코머티리얼즈  126600
기업코드 데이터 타입: object

KOSDAQ 종목코드 샘플:
     종목명    종목코드    시장구분
0     3S  060310  KOSDAQ
1    APS  054620  KOSDAQ
2  AP시스템  265520  KOSDAQ
3   AP위성  211270  KOSDAQ
4   BF랩스  139050  KOSDAQ
종목코드 데이터 타입: object


In [106]:
# ============================================
# 셀 5: 최소 정규화 및 변환 기업 추적
# ============================================

# 1. 원본 기업코드 백업 (비교용)
df_excel['기업코드_원본'] = df_excel['기업코드'].astype(str)

# 2. 기업코드 정규화 (공백 제거 + 6자리 미만 숫자에만 앞자리 '0' 보완)
# ※ 문자열 공백을 제거한 뒤, 6자리 미만일 때만 앞에 '0'을 채워 최소한으로만 변경합니다.
df_excel['기업코드'] = df_excel['기업코드_원본'].str.strip().str.zfill(6)

# KOSDAQ / KOSPI 종목코드도 동일 적용
df_kosdaq['종목코드'] = df_kosdaq['종목코드'].astype(str).str.strip().str.zfill(6)
df_kospi['종목코드'] = df_kospi['종목코드'].astype(str).str.strip().str.zfill(6)

# 3. 코드 값이 변경된(정규화된) 기업 추출
changed_mask = df_excel['기업코드_원본'] != df_excel['기업코드']
df_changed = df_excel[changed_mask].copy()

# ============================================
# 정규화 결과 리포트 출력
# ============================================
print("=" * 60)
print(f"📊 기업코드 정규화 현황 (전체 {len(df_excel)}개 기업 중)")
print("=" * 60)
print(f"  • 변환 없이 그대로 유지된 기업 : {len(df_excel) - len(df_changed)}개")
print(f"  • 정규화(변환)된 기업          : {len(df_changed)}개")

print("\n" + "=" * 60)
print(f"🔍 정규화된 기업 전체 목록 ({len(df_changed)}개)")
print("=" * 60)

if len(df_changed) > 0:
    # 출력할 주요 컬럼 지정 (존재하는 컬럼만 선택)
    cols = [col for col in ['NO', 'No', '기업명', '기업코드_원본', '기업코드'] if col in df_changed.columns]
    
    # 변경된 전체 목록 출력
    print(df_changed[cols].rename(columns={
        '기업코드_원본': '변환 전(원본)',
        '기업코드': '변환 후(6자리)'
    }).to_string(index=False))
else:
    print("✨ 변환된 기업이 없습니다. 모든 기업코드가 원본 상태 그대로 유지되었습니다.")

# 4. 검증 완료 후 비교용 백업 컬럼 삭제
df_excel = df_excel.drop(columns=['기업코드_원본'])

📊 기업코드 정규화 현황 (전체 1091개 기업 중)
  • 변환 없이 그대로 유지된 기업 : 1091개
  • 정규화(변환)된 기업          : 0개

🔍 정규화된 기업 전체 목록 (0개)
✨ 변환된 기업이 없습니다. 모든 기업코드가 원본 상태 그대로 유지되었습니다.


In [107]:
# ============================================
# 셀 6: 매칭 검증 (첫 번째 기업으로 테스트)
# ============================================
test_code = df_excel['기업코드'].iloc[0]
test_name = df_excel['기업명'].iloc[0]
 
print("\n" + "=" * 60)
print(f"매칭 검증 - 첫 번째 기업 테스트")
print("=" * 60)
print(f"기업코드: {test_code}")
print(f"기업명: {test_name}")
 
# KOSDAQ 확인
kosdaq_match = df_kosdaq[df_kosdaq['종목코드'] == test_code]
if len(kosdaq_match) > 0:
    print(f"\n✅ KOSDAQ에서 찾음!")
    print(f"   종목명: {kosdaq_match['종목명'].values[0]}")
    print(f"   시장구분: {kosdaq_match['시장구분'].values[0]}")
else:
    print(f"\n❌ KOSDAQ에서 못 찾음")
 
# KOSPI 확인
kospi_match = df_kospi[df_kospi['종목코드'] == test_code]
if len(kospi_match) > 0:
    print(f"\n✅ KOSPI에서 찾음!")
    print(f"   종목명: {kospi_match['종목명'].values[0]}")
    print(f"   시장구분: {kospi_match['시장구분'].values[0]}")
else:
    print(f"\n❌ KOSPI에서 못 찾음")


매칭 검증 - 첫 번째 기업 테스트
기업코드: 095570
기업명: AJ네트웍스

❌ KOSDAQ에서 못 찾음

✅ KOSPI에서 찾음!
   종목명: AJ네트웍스
   시장구분: KOSPI


In [108]:
# ============================================
# 셀 7: KOSDAQ + KOSPI 병합
# ============================================
df_market_data = pd.concat([df_kosdaq, df_kospi], ignore_index=True)
 
print("\n✓ KOSDAQ + KOSPI 병합 완료")
print(f"  총 {len(df_market_data)}개 종목")
print(f"\n시장 구분별 개수:")
print(df_market_data['시장구분'].value_counts())


✓ KOSDAQ + KOSPI 병합 완료
  총 2788개 종목

시장 구분별 개수:
시장구분
KOSDAQ    1830
KOSPI      958
Name: count, dtype: int64


In [109]:
# ============================================
# 셀 8: JOIN 실행 (기업코드 기준)
# ============================================
"""
기업코드 기준으로 LEFT JOIN 수행
가져올 컬럼: 시장구분, 업종명, 종가, 대비, 등락률, 시가총액
"""
 
join_columns = ['시장구분', '업종명', '종가', '대비', '등락률', '시가총액']
 
# 기업코드 기준으로 LEFT JOIN
df_result = df_excel.merge(
    df_market_data[['종목코드'] + join_columns],
    left_on='기업코드',
    right_on='종목코드',
    how='left'
)
 
# 중복된 종목코드 컬럼 제거 (기업코드만 유지)
if '종목코드' in df_result.columns:
    df_result = df_result.drop('종목코드', axis=1)
 
print("✓ JOIN 완료")
print(f"  결과 행 수: {len(df_result)}")


✓ JOIN 완료
  결과 행 수: 1091


In [110]:
# ============================================
# 셀 9: 결과 확인
# ============================================
print("\n" + "=" * 60)
print("조인 결과 확인")
print("=" * 60)

# 매칭 현황
matched = df_result['시장구분'].notna().sum()
unmatched = df_result['시장구분'].isna().sum()

print(f"\n✅ 매칭된 종목: {matched}개")
print(f"❌ 미매칭 종목: {unmatched}개")
print(f"   매칭률: {(matched/len(df_result)*100):.1f}%")

# 출력할 컬럼 지정 (실제 존재하는 컬럼만 안전하게 선별)
target_cols = ['NO', 'No', '순번', '기업명', '기업코드', '시장구분', '업종명', '종가', '시가총액']
sample_cols = [col for col in target_cols if col in df_result.columns]

# 샘플 데이터 (매칭된 것)
print("\n조인된 데이터 샘플 (처음 5행):")
matched_data = df_result[df_result['시장구분'].notna()].head(5)
print(matched_data[sample_cols].to_string(index=False))

# 미매칭 데이터 확인
if unmatched > 0:
    print(f"\n⚠️ 미매칭 종목 샘플 (처음 5개):")
    unmatch_cols = [col for col in ['NO', 'No', '기업명', '기업코드'] if col in df_result.columns]
    unmatch_data = df_result[df_result['시장구분'].isna()].head(5)
    print(unmatch_data[unmatch_cols].to_string(index=False))


조인 결과 확인

✅ 매칭된 종목: 1018개
❌ 미매칭 종목: 73개
   매칭률: 93.3%

조인된 데이터 샘플 (처음 5행):
       기업명   기업코드   시장구분   업종명       종가         시가총액
    AJ네트웍스 095570  KOSPI 일반서비스   4660.0 2.108779e+11
     AK홀딩스 006840  KOSPI  기타금융   8710.0 1.153863e+11
       BGF 027410  KOSPI  기타금융   3920.0 3.752098e+11
    BGF리테일 282330  KOSPI    유통 104800.0 1.811353e+12
BGF에코머티리얼즈 126600 KOSDAQ    화학   3660.0 2.297269e+11

⚠️ 미매칭 종목 샘플 (처음 5개):
    기업명   기업코드
 BNK캐피탈 801190
BNK투자증권 014876
 DB생명보험 818158
 HD현대미포 010620
 KB손해보험 002550


In [111]:
# ============================================
# 셀 10: CSV로 저장
# ============================================
 
# 1. 전체 결과 저장
df_result.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n✓ 전체 결과 저장 완료")
print(f"  경로: {output_file}")
print(f"  행 수: {len(df_result)}")
 
# 2. KOSDAQ만 저장
df_result_kosdaq = df_result[df_result['시장구분'] == 'KOSDAQ'].copy()
kosdaq_output = output_file.replace('.csv', '_KOSDAQ.csv')
df_result_kosdaq.to_csv(kosdaq_output, index=False, encoding='utf-8-sig')
print(f"\n✓ KOSDAQ 분리 저장 ({len(df_result_kosdaq)}개 종목)")
print(f"  경로: {kosdaq_output}")
 
# 3. KOSPI만 저장
df_result_kospi = df_result[df_result['시장구분'] == 'KOSPI'].copy()
kospi_output = output_file.replace('.csv', '_KOSPI.csv')
df_result_kospi.to_csv(kospi_output, index=False, encoding='utf-8-sig')
print(f"\n✓ KOSPI 분리 저장 ({len(df_result_kospi)}개 종목)")
print(f"  경로: {kospi_output}")
 
print("\n" + "=" * 60)
print("✅ 모든 작업 완료!")
print("=" * 60)


✓ 전체 결과 저장 완료
  경로: /Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_조인.csv
  행 수: 1091

✓ KOSDAQ 분리 저장 (219개 종목)
  경로: /Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_조인_KOSDAQ.csv

✓ KOSPI 분리 저장 (799개 종목)
  경로: /Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_조인_KOSPI.csv

✅ 모든 작업 완료!


In [112]:
# ============================================
# 셀 11: 최종 검증 및 통계
# ============================================
print("\n" + "=" * 60)
print("최종 데이터 검증")
print("=" * 60)
 
# 기업코드 무결성 확인
print(f"\n✓ 기업코드 데이터 타입: {df_result['기업코드'].dtype}")
print(f"✓ 기업코드 누락: {df_result['기업코드'].isna().sum()}개")
print(f"✓ 기업코드 샘플: {df_result['기업코드'].head(5).tolist()}")
 
# 최종 통계
print(f"\n시장별 조인 결과:")
print(f"  KOSDAQ: {len(df_result_kosdaq)}개 ({len(df_result_kosdaq)/len(df_result)*100:.1f}%)")
print(f"  KOSPI: {len(df_result_kospi)}개 ({len(df_result_kospi)/len(df_result)*100:.1f}%)")
print(f"  미매칭: {unmatched}개 ({unmatched/len(df_result)*100:.1f}%)")
print(f"  총계: {len(df_result)}개 (원본: {len(df_excel)}개)")
 
print(f"\n저장된 컬럼:")
for i, col in enumerate(df_result.columns, 1):
    print(f"  {i}. {col}")
 


최종 데이터 검증

✓ 기업코드 데이터 타입: object
✓ 기업코드 누락: 0개
✓ 기업코드 샘플: ['095570', '006840', '027410', '282330', '126600']

시장별 조인 결과:
  KOSDAQ: 219개 (20.1%)
  KOSPI: 799개 (73.2%)
  미매칭: 73개 (6.7%)
  총계: 1091개 (원본: 1091개)

저장된 컬럼:
  1. 기업명
  2. 기업코드
  3. ESG등급
  4. 환경
  5. 사회
  6. 지배구조
  7. 평가년도
  8. ESG 등급조정
  9. 환경 등급조정
  10. 사회 등급조정
  11. 지배구조 등급조정
  12. 시장구분
  13. 업종명
  14. 종가
  15. 대비
  16. 등락률
  17. 시가총액


---
## 2. 데이터셋 추가 조인
-> 최근 5개년 평가등급 추가 (2021~2025)

-> 파일명: KCGS_평가등급_2021-2025_Long

In [113]:
import pandas as pd

# 1. 파일 경로 설정
excel_file = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/전체/99_이전데이터셋/(2021~2025)_평가등급(2026).xlsx'
csv_file = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_조인.csv'
output_file = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv'

# 2. 파일 읽기 (기업코드는 문자열 6자리로 정규화)
df_excel = pd.read_excel(excel_file, dtype={'기업코드': str})
df_csv = pd.read_csv(csv_file, dtype={'기업코드': str})

# 코드 정규화 (공백 제거 및 6자리 0 채우기)
df_excel['기업코드'] = df_excel['기업코드'].astype(str).str.strip().str.zfill(6)
df_csv['기업코드'] = df_csv['기업코드'].astype(str).str.strip().str.zfill(6)

# 평가년도 정수형 변환
df_excel['평가년도'] = pd.to_numeric(df_excel['평가년도'], errors='coerce').fillna(0).astype(int)
df_csv['평가년도'] = pd.to_numeric(df_csv['평가년도'], errors='coerce').fillna(0).astype(int)

# 3. CSV에서 기업별 '시장구분', '업종명' 매핑 정보 추출
market_info = df_csv[['기업코드', '시장구분', '업종명']].drop_duplicates(subset=['기업코드'])

# 4. 엑셀 데이터에 시장구분, 업종명 조인 (LEFT JOIN)
df_result = df_excel.merge(market_info, on='기업코드', how='left')

# 5. 2025년도 주가/시총 금융 데이터 매핑 (2021~2024년도는 자동으로 NaN 처리)
financial_cols = ['종가', '대비', '등락률', '시가총액']
csv_financials = df_csv[['기업코드', '평가년도'] + financial_cols]

df_result = df_result.merge(csv_financials, on=['기업코드', '평가년도'], how='left')

# 6. 컬럼 순서 재정렬
base_cols = ['기업명', '기업코드', '평가년도', 'ESG등급', '환경', '사회', '지배구조',
             'ESG 등급조정', '환경 등급조정', '사회 등급조정', '지배구조 등급조정',
             '시장구분', '업종명', '종가', '대비', '등락률', '시가총액']

# 실제 존재하는 컬럼만 선별하여 정리
existing_cols = [col for col in base_cols if col in df_result.columns]
df_result = df_result[existing_cols]

# 7. 정렬: 기업명(오름차순) -> 평가년도(내림차순: 2025 -> 2021)
df_result = df_result.sort_values(by=['기업명', '평가년도'], ascending=[True, False])

# 8. CSV 파일로 저장
df_result.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✓ 데이터 합성 및 정렬 완료!")
print(f"✓ 저장 경로: {output_file}")
print(f"✓ 총 행 수: {len(df_result):,}개")

# 결과 샘플 확인
print("\n[상위 10개 행 샘플]")
print(df_result[['기업명', '기업코드', '평가년도', 'ESG등급', '시장구분', '업종명', '종가', '시가총액']].head(10).to_string(index=False))

✓ 데이터 합성 및 정렬 완료!
✓ 저장 경로: /Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv
✓ 총 행 수: 5,245개

[상위 10개 행 샘플]
   기업명   기업코드  평가년도 ESG등급  시장구분   업종명     종가         시가총액
AJ네트웍스 095570  2025    B+ KOSPI 일반서비스 4660.0 2.108779e+11
AJ네트웍스 095570  2024    B+ KOSPI 일반서비스    NaN          NaN
AJ네트웍스 095570  2023    B+ KOSPI 일반서비스    NaN          NaN
AJ네트웍스 095570  2022    B+ KOSPI 일반서비스    NaN          NaN
AJ네트웍스 095570  2021     B KOSPI 일반서비스    NaN          NaN
 AK홀딩스 006840  2025     A KOSPI  기타금융 8710.0 1.153863e+11
 AK홀딩스 006840  2024     A KOSPI  기타금융    NaN          NaN
 AK홀딩스 006840  2023     A KOSPI  기타금융    NaN          NaN
 AK홀딩스 006840  2022    B+ KOSPI  기타금융    NaN          NaN
 AK홀딩스 006840  2021    B+ KOSPI  기타금융    NaN          NaN


---
## 3. 산업 리서치 EDA
- [기준 데이터셋] KCGS_평가등급_2021-2025_Long.csv

##### 3-1. 작업 설정
- 플러그인 설치
- 폰트 설정
- 데이터셋 불러오기

In [114]:
# ============================================
# 셀 1: 라이브러리 임포트 + Mac 한글 폰트 설정
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
 
# ★ Mac 한글 폰트 설정 (필수)
import matplotlib.font_manager as fm
 
# 【옵션 선택】 다음 중 하나를 선택해서 주석 해제
# ────────────────────────────────────────────
 
# # ✅ 옵션 1: Noto Sans CJK JP (현재 추천 - 설치 불필요)
# plt.rcParams['font.family'] = 'Noto Sans CJK JP'
 
# # ❌ 옵션 2: Noto Sans CJK KR (설치 필요)
# fm.fontManager.addfont('/Library/Fonts/NotoSansCJKkr-Regular.otf')
# plt.rcParams['font.family'] = 'Noto Sans CJK KR'
 
# # ❌ 옵션 3: Apple SD Gothic Neo (Mac 기본)
# fm.fontManager.addfont('/Library/Fonts/AppleSDGothicNeo.ttc')
# Apple SD Gothic Neo 설정
plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스(-) 기호 깨짐 방지
# fig_plotly.update_layout(font_family="Apple SD Gothic Neo")


 
print("✓ 모든 라이브러리 로드 완료")
print(f"✓ 한글 폰트 설정: {plt.rcParams['font.family']}")

✓ 모든 라이브러리 로드 완료
✓ 한글 폰트 설정: ['Apple SD Gothic Neo']


In [115]:
!pip install koreanize-matplotlib

import matplotlib.pyplot as plt
import koreanize_matplotlib  # 이 한 줄로 Mac에서도 한글 패치 완료!

# 이제 plt.rcParams['font.family'] 설정을 따로 안 하셔도 됩니다.

In [116]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Mac 내장 애플산돌고딕 폰트 파일 경로 직접 지정
font_path = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'  # 또는 '/System/Library/Fonts/AppleSDGothicNeo.ttc'

# 폰트 등록
fm.fontManager.addfont(font_path)
font_prop = fm.FontProperties(fname=font_path)

# 기본 폰트로 적용
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

print(f"✓ 적용된 폰트: {plt.rcParams['font.family']}")

✓ 적용된 폰트: ['AppleGothic']


In [117]:
# ============================================
# 셀 2: 데이터 로드 및 기본 정보
# ============================================
# 파일 경로 설정
csv_file = r'/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv'
 
# 데이터 로드
df = pd.read_csv(csv_file)
 
print("✓ 데이터 로드 완료")
print(f"\n데이터셋 정보:")
print(f"  - 행 수: {len(df):,}개 기업")
print(f"  - 컬럼 수: {len(df.columns)}개")
print(f"  - 시장구분: {df['시장구분'].unique()}")
print(f"\n컬럼명: {df.columns.tolist()}")

✓ 데이터 로드 완료

데이터셋 정보:
  - 행 수: 5,245개 기업
  - 컬럼 수: 17개
  - 시장구분: ['KOSPI' nan 'KOSDAQ']

컬럼명: ['기업명', '기업코드', '평가년도', 'ESG등급', '환경', '사회', '지배구조', 'ESG 등급조정', '환경 등급조정', '사회 등급조정', '지배구조 등급조정', '시장구분', '업종명', '종가', '대비', '등락률', '시가총액']


##### 3-2. 최근 3개년(2023~2025) ESG 종합등급 분포 추이

In [118]:
import plotly.io as pio

# VS Code 전용 공식 렌더러 이름
pio.renderers.default = "vscode"

In [119]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 데이터 로드
df = pd.read_csv('/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv')

# 2. 등급 정렬 순서 정의 (S부터 등급없음까지)
grade_order = ['S', 'A+', 'A', 'B+', 'B', 'C', 'D', '등급없음']

# 3. 2023~2025년 데이터 필터링 및 집계
df_3years = df[df['평가년도'].isin([2023, 2024, 2025])].copy()

# 연도 컬럼을 문자열로 변환하여 범주형(Discrete) 색상 적용
df_3years['평가년도'] = df_3years['평가년도'].astype(str)

esg_counts = df_3years.groupby(['평가년도', 'ESG등급']).size().reset_index(name='기업수')

# 4. 차트 생성 (barmode='group'으로 세로 나란히 배치)
fig = px.bar(
    esg_counts,
    x='ESG등급',
    y='기업수',
    color='평가년도',
    barmode='group',  # 세로 나란히 세우기 (Non-stacked)
    category_orders={'ESG등급': grade_order},
    title='<b>[1] 최근 3개년(2023~2025) ESG 종합등급 분포 추이</b>',
    labels={'기업수': '기업 수 (개)', 'ESG등급': 'ESG 종합등급', '평가년도': '연도'},
    # 소프트하고 채도가 낮은 파스텔/뮤트 컬러 팔레트 직접 지정
    color_discrete_map={
        '2023': '#A8C3A8',  # 차분한 세이지 그린
        '2024': '#87A8A8',  # Muted 블루그린
        '2025': '#6B8E93'   # Muted 딥 블루그린
    }
)

# 5. 레이아웃 세부 설정 (텍스트 및 배경)
fig.update_traces(
    texttemplate='%{y}',           # 막대 위에 기업 수 숫 표시
    textposition='outside'         # 막대 바깥 상단 표기
)

fig.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    legend_title_text='평가년도',
    font=dict(color='#4A4A4A'),    # 텍스트 컬러를 눈이 편한 차콜색으로 설정
    yaxis=dict(showgrid=True, gridcolor='#F0F0F0'), # 가로 격자선 연하게
    bargap=0.2,                   # 등급 간 간격
    bargroupgap=0.1               # 연도별 막대 간 간격
)

fig.show()

##### 3-3. 최근 3개년 E(환경), S(사회), G(지배구조) 항목별 등급 분포

In [120]:
grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']

df_e = df[(df['평가년도'].isin([2023, 2024, 2025])) & (df['환경'].isin(grade_order))].copy()
df_e['평가년도'] = df_e['평가년도'].astype(str)

e_counts = df_e.groupby(['평가년도', '환경']).size().reset_index(name='기업수')

# E (환경) 개별 차트 생성
fig_e = px.bar(
    e_counts,
    x='환경',
    y='기업수',
    color='평가년도',
    barmode='group',
    category_orders={'환경': grade_order},
    title='<b>[2-E] 최근 3개년 E(환경) 등급 분포 추이 (2023~2025)</b>',
    labels={'기업수': '기업 수 (개)', '환경': '환경 등급', '평가년도': '연도'},
    color_discrete_map={
        '2023': '#D0E8C5',  # 연한 세이지 연두
        '2024': '#A0C878',  # 파스텔 연두
        '2025': '#5F9EA0'   # 차분한 딥 올리브/연두
    }
)

fig_e.update_traces(texttemplate='%{y}', textposition='outside')
fig_e.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    bargap=0.2,
    bargroupgap=0.1
)

fig_e.show()

In [121]:
df_s = df[(df['평가년도'].isin([2023, 2024, 2025])) & (df['사회'].isin(grade_order))].copy()
df_s['평가년도'] = df_s['평가년도'].astype(str)

s_counts = df_s.groupby(['평가년도', '사회']).size().reset_index(name='기업수')

# S (사회) 개별 차트 생성
fig_s = px.bar(
    s_counts,
    x='사회',
    y='기업수',
    color='평가년도',
    barmode='group',
    category_orders={'사회': grade_order},
    title='<b>[2-S] 최근 3개년 S(사회) 등급 분포 추이 (2023~2025)</b>',
    labels={'기업수': '기업 수 (개)', '사회': '사회 등급', '평가년도': '연도'},
    color_discrete_map={
        '2023': '#D0E8F2',  # 연한 하늘색
        '2024': '#79D7F2',  # 파스텔 하늘색
        '2025': '#457B9D'   # 차분한 Muted 블루
    }
)

fig_s.update_traces(texttemplate='%{y}', textposition='outside')
fig_s.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    bargap=0.2,
    bargroupgap=0.1
)

fig_s.show()

In [122]:
df_g = df[(df['평가년도'].isin([2023, 2024, 2025])) & (df['지배구조'].isin(grade_order))].copy()
df_g['평가년도'] = df_g['평가년도'].astype(str)

g_counts = df_g.groupby(['평가년도', '지배구조']).size().reset_index(name='기업수')

# G (지배구조) 개별 차트 생성
fig_g = px.bar(
    g_counts,
    x='지배구조',
    y='기업수',
    color='평가년도',
    barmode='group',
    category_orders={'지배구조': grade_order},
    title='<b>[2-G] 최근 3개년 G(지배구조) 등급 분포 추이 (2023~2025)</b>',
    labels={'기업수': '기업 수 (개)', '지배구조': '지배구조 등급', '평가년도': '연도'},
    color_discrete_map={
        '2023': '#C5D3E8',  # 연한 페일 네이비
        '2024': '#8294C4',  # 소프트 인디고
        '2025': '#1D2A44'   # 차분한 딥 네이비
    }
)

fig_g.update_traces(texttemplate='%{y}', textposition='outside')
fig_g.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    bargap=0.2,
    bargroupgap=0.1
)

fig_g.show()

##### 3-4. 시장구분별(KOSPI vs KOSDAQ) 총 상장수 및 등급별 분포

In [123]:
grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']

df_2025 = df[df['평가년도'] == 2025].copy()

# 2. 시장구분 결측치(NaN)를 '미등록(미분류)'으로 채우기
df_2025['시장구분_분류'] = df_2025['시장구분'].fillna('미등록(미분류)')

# --- [3] 총 등록 기업 수 중 시장구분별 비중 비교 (도넛 파이차트) ---
market_counts = df_2025['시장구분_분류'].value_counts().reset_index()
market_counts.columns = ['시장구분', '기업수']

fig3 = px.pie(
    market_counts,
    names='시장구분',
    values='기업수',
    hole=0.4,
    title='<b>[3] 2025년 기준 전체 ESG 등록 기업의 시장구분별(KOSPI, KOSDAQ, 미등록) 비중</b>',
    color='시장구분',
    color_discrete_map={
        'KOSPI': '#003366',       # 딥 네이비
        'KOSDAQ': '#FF6600',      # 오렌지
        '미등록(미분류)': '#D3D3D3' # 차분한 그레이
    }
)

# 퍼센트(%) 및 수치(value) 함께 표기
fig3.update_traces(
    textinfo='label+value+percent',
    textfont_size=13,
    hovertemplate='<b>%{label}</b><br>기업 수: %{value}개<br>비율: %{percent}'
)
fig3.update_layout(font_family='Apple SD Gothic Neo', template='plotly_white')
fig3.show()

# --- [4] 시장구분별 ESG 등급 분포 (미등록 포함) ---
market_esg = df_2025.groupby(['시장구분_분류', 'ESG등급']).size().reset_index(name='기업수')
# A+ ~ D 등급 데이터만 선별
market_esg = market_esg[market_esg['ESG등급'].isin(grade_order)]

fig4 = px.bar(
    market_esg,
    x='ESG등급',
    y='기업수',
    color='시장구분_분류',
    barmode='group',
    category_orders={'ESG등급': grade_order},
    title='<b>[4] 2025년 기준 시장구분별 ESG 등급 분포 비교</b>',
    labels={'기업수': '기업 수 (개)', 'ESG등급': 'ESG 종합등급', '시장구분_분류': '시장구분'},
    color_discrete_map={
        'KOSPI': '#003366',
        'KOSDAQ': '#FF6600',
        '미등록(미분류)': '#D3D3D3'
    }
)

fig4.update_traces(texttemplate='%{y}', textposition='outside')
fig4.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    bargap=0.2,
    bargroupgap=0.1
)
fig4.show()

##### 3-5. 전체 업종별 기업 수 및 TOP 10 주요 산업군

In [124]:
df['업종명_채움'] = df['업종명'].fillna('미분류(없음)')

# 2. 최신(2025년) 기준 기업별 업종 집계
df_2025 = df[df['평가년도'] == 2025].copy()
sector_counts = df_2025['업종명_채움'].value_counts().reset_index()
sector_counts.columns = ['업종명', '기업수']

# 상위 10개 업종 추출
top10_sectors = sector_counts.head(10)['업종명'].tolist()

# [1&2] 상장 기업수 기준 TOP 10 주요 산업군 시각화
fig1 = px.bar(
    sector_counts.head(15),  # 상위 15개 표시 (TOP 10 명확 구분)
    x='기업수',
    y='업종명',
    orientation='h',
    title='<b>[1] 상장 기업수 기준 TOP 10 주요 산업군 현황 (미분류 포함)</b>',
    labels={'기업수': '기업 수 (개)', '업종명': '산업군'},
    color='업종명',
    # TOP 10 업종 강조 색상
    # color_discrete_sequence=px.colors.qualitative.Muted
)

fig1.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    showlegend=False
)
fig1.update_traces(texttemplate='%{x}개', textposition='outside')
fig1.show()

print("✓ TOP 10 주요 산업군 (기업 수):")
for idx, row in sector_counts.head(10).iterrows():
    print(f" {idx+1}위. {row['업종명']}: {row['기업수']}개사")

✓ TOP 10 주요 산업군 (기업 수):
 1위. 화학: 125개사
 2위. 전기·전자: 101개사
 3위. 유통: 74개사
 4위. 미분류(없음): 73개사
 5위. 기타금융: 70개사
 6위. 제약: 69개사
 7위. 운송장비·부품: 64개사
 8위. 금속: 63개사
 9위. 기계·장비: 61개사
 10위. 일반서비스: 55개사


##### 3-6. TOP 10 주요 산업군의 최근 3개년(2023~2025) ESG 등급 추이

In [125]:
grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']

# TOP 10 산업군 & 최근 3개년 데이터 필터링
df_top10_3y = df[(df['평가년도'].isin([2023, 2024, 2025])) & 
                 (df['업종명_채움'].isin(top10_sectors)) & 
                 (df['ESG등급'].isin(grade_order))].copy()

df_top10_3y['평가년도'] = df_top10_3y['평가년도'].astype(str)

sector_esg_3y = df_top10_3y.groupby(['업종명_채움', '평가년도', 'ESG등급']).size().reset_index(name='기업수')

# [3] TOP 10 주요 산업군의 최근 3개년 ESG 등급 분포
fig3 = px.bar(
    sector_esg_3y,
    x='ESG등급',
    y='기업수',
    color='평가년도',
    facet_col='업종명_채움',
    facet_col_wrap=5,  # 5개씩 2줄 배치
    barmode='group',
    category_orders={'ESG등급': grade_order, '업종명_채움': top10_sectors},
    title='<b>[3] TOP 10 주요 산업군별 최근 3개년(2023~2025) ESG 등급 분포 변화</b>',
    labels={'기업수': '기업 수', 'ESG등급': 'ESG 등급', '평가년도': '연도'},
    color_discrete_map={'2023': '#A8C3A8', '2024': '#87A8A8', '2025': '#6B8E93'}
)

fig3.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    legend_title_text='평가년도'
)
fig3.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1])) # 서브플롯 타이틀 정리
fig3.show()

##### 3-7. TOP 10 주요 산업군별 ESG 등급 수준 분포

In [126]:
# 1. 등급을 점수로 수치화 (박스플롯 계산용)
grade_to_score = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}
score_to_grade = {6: 'A+', 5: 'A', 4: 'B+', 3: 'B', 2: 'C', 1: 'D'}

df_box = df_top10_3y[df_top10_3y['평가년도'] == '2025'].copy() # 2025년 최신 기준
df_box['ESG_점수'] = df_box['ESG등급'].map(grade_to_score)

# [4] TOP 10 주요 산업군별 ESG 등급 박스플롯 시각화
fig4 = px.box(
    df_box,
    x='업종명_채움',
    y='ESG_점수',
    color='업종명_채움',
    points='all',  # 전체 기업 개별 점수 도트 표현
    category_orders={'업종명_채움': top10_sectors},
    title='<b>[4] 2025년 TOP 10 주요 산업군별 ESG 등급 수준 및 편차 분포 (Boxplot)</b>',
    labels={'업종명_채움': '주요 산업군', 'ESG_점수': 'ESG 등급 수준'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# Y축 수치를 다시 ESG 문자 등급 표기로 변경
fig4.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    showlegend=False,
    yaxis=dict(
        tickmode='array',
        tickvals=[1, 2, 3, 4, 5, 6],
        ticktext=['D', 'C', 'B', 'B+', 'A', 'A+']
    )
)

fig4.show()

# 기본 통계량 출력 (요약)
print("\n✓ 2025년 TOP 10 산업군별 ESG 등급 점수 기본 통계 요약 (6=A+ ~ 1=D):")
stats_summary = df_box.groupby('업종명_채움')['ESG_점수'].agg(
    기업수='count',
    평균점수='mean',
    중앙값='median',
    최고점수='max',
    최저점수='min'
).reset_index().sort_values(by='기업수', ascending=False)

print(stats_summary.to_string(index=False))


✓ 2025년 TOP 10 산업군별 ESG 등급 점수 기본 통계 요약 (6=A+ ~ 1=D):
 업종명_채움  기업수     평균점수  중앙값  최고점수  최저점수
     화학  125 2.816000  2.0     6     1
  전기·전자  101 2.574257  2.0     6     1
     유통   74 2.648649  2.0     6     1
   기타금융   70 3.785714  4.0     6     1
     제약   69 2.898551  2.0     5     1
운송장비·부품   64 2.359375  2.0     6     1
     금속   63 2.238095  2.0     6     1
  기계·장비   61 2.262295  2.0     6     1
  일반서비스   55 3.054545  3.0     5     1
미분류(없음)    8 2.250000  1.0     5     1


### [인사이트]
1. 상장 기업 수 TOP 3: 화학(125개사), 전기·전자(101개사), 유통(74개사) 순

2. 산업별 ESG 성숙도: 기타금융(중앙값: B+ 등급 수준)이 상대적으로 높은 ESG 수행 수준 보임.
그러나, 금속/기계·장비/운송장비·부품 계열은 중앙값이 C 등급 수준으로 하위 등급 기업 비중이 높은 편

-> 산업 정하는 것이 필요해 보임

---
## 4. ESG 등급과 시가총액 상관관계 EDA
- [기준 데이터셋] KCGS_평가등급_2021-2025_Long.csv

#####  [가설] ESG 평가 등급이 높으면 기업 가치도 높다.
4-1. TOP 5 주요 업종별 시가총액 기초 통계량

In [127]:
df_2025 = df[(df['평가년도'] == 2025) & (df['시가총액'].notna())].copy()

# 시가총액 단위 변환 (조 원 단위 실수값 생성)
df_2025['시가총액_조원'] = df_2025['시가총액'] / 1e12

# 한글 표기 변환 함수
def format_korean(val_raw):
    if pd.isna(val_raw) or val_raw == 0:
        return '0원'
    val = int(round(val_raw))
    trillion = val // 1000000000000
    hundred_million = (val % 1000000000000) // 100000000
    res = []
    if trillion > 0:
        res.append(f"{trillion:,}조")
    if hundred_million > 0:
        res.append(f"{hundred_million:,}억")
    return ' '.join(res) + ' 원' if res else f"{val:,}원"

df_2025['시가총액_한글'] = df_2025['시가총액'].apply(format_korean)

# 2. TOP 5 업종 추출
top5_sectors = df_2025['업종명'].value_counts().head(5).index.tolist()
df_top5 = df_2025[df_2025['업종명'].isin(top5_sectors)].copy()

# 3. 박스플롯 시각화
fig = px.box(
    df_top5,
    x='업종명',
    y='시가총액_조원',
    color='업종명',
    points='all',
    log_y=True,  # 편차가 커서 로그 축 유지
    hover_data={'기업명': True, '시가총액_한글': True, '시가총액_조원': False},
    category_orders={'업종명': top5_sectors},
    title='<b>[1] TOP 5 주요 업종별 기업 시가총액 분포</b>',
    labels={'업종명': '주요 업종', '시가총액_조원': '시가총액 (조 원 단위)'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    showlegend=False
)
fig.show()

📊 TOP 5 업종 시가총액 기본 통계량 요약 (억원 기준)

1. 전기·전자 (101개사) | 평균: 150,054억 원 | 중앙값: 6,126억 원 | 최고: 7,097,646억 원 | 총합: 15,155,470억 원

2. 기타금융 (70개사) | 평균: 54,476억 원 | 중앙값: 8,074억 원 | 최고: 486,081억 원 | 총합: 3,813,326억 원

3. 제약 (69개사) | 평균: 31,247억 원 | 중앙값: 4,910억 원 | 최고: 784,632억 원 | 총합: 2,156,009억 원

4. 화학 (125개사) | 평균: 11,998억 원 | 중앙값: 3,192억 원 | 최고: 235,073억 원 | 총합: 1,499,705억 원

5. 유통 (74개사) | 평균: 12,645억 원 | 중앙값: 2,525억 원 | 최고: 407,094억 원 | 총합: 935,706억 원

##### 4-2. ESG 우수 기업(A+, A) vs 일반 기업(B+, B, C, D) 시가총액 분포 비교

In [128]:
# ESG 그룹 분류 (A+/A 그룹 vs B/C/D 그룹)
def classify_esg(grade):
    if grade in ['A+', 'A']:
        return '우수 그룹 (A+, A)'
    elif grade in ['B+', 'B', 'C', 'D']:
        return '일반/하위 그룹 (B~D)'
    return None

df_2025['ESG_그룹'] = df_2025['ESG등급'].apply(classify_esg)
df_esg_filtered = df_2025[df_2025['ESG_그룹'].notna()].copy()

# [2] ESG 그룹별 시가총액 분포 바이올린/박스플롯 시각화
fig2 = px.box(
    df_esg_filtered,
    x='ESG_그룹',
    y='시가총액_조원',
    color='ESG_그룹',
    points='all',
    log_y=True,  # 시가총액 격차가 커 로그 스케일 활용
    title='<b>[2] ESG 등급 그룹별 시가총액 분포 비교 (A+/A vs B/C/D)</b>',
    labels={'ESG_그룹': 'ESG 등급 그룹', '시가총액_억원': '시가총액 (억원)'},
    color_discrete_map={
        '우수 그룹 (A+, A)': '#457B9D',      # Muted 딥 블루
        '일반/하위 그룹 (B~D)': '#E07A5F'    # Muted 테라코타 오렌지
    }
)

fig2.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    showlegend=False,
    yaxis=dict(tickformat=',.0f')
)
fig2.show()

##### 4-3. ESG 등급 및 세부항목별 상관관계

In [129]:
df_2025 = df[(df['평가년도'] == 2025) & (df['시가총액'].notna())].copy()

# 2. 등급을 수치 점수로 변환 (A+=6 ~ D=1)
grade_map = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}
metrics = ['ESG등급', '환경', '사회', '지배구조']

for col in metrics:
    df_2025[f'{col}_점수'] = df_2025[col].map(grade_map)

# 3. 스피어만 상관계수 계산
corr_cols = ['ESG등급_점수', '환경_점수', '사회_점수', '지배구조_점수', '시가총액']
corr_matrix = df_2025[corr_cols].corr(method='spearman')

# 라벨 명칭 변경
renamed_labels = ['ESG 종합', 'E (환경)', 'S (사회)', 'G (지배구조)', '시가총액']
corr_matrix.columns = renamed_labels
corr_matrix.index = renamed_labels

# [3] 상관관계 히트맵 시각화
fig_heatmap = px.imshow(
    corr_matrix.round(3),
    text_auto=True,
    color_continuous_scale='Blues',
    title='<b>[3] ESG 등급과 시가총액 간 상관계수</b>',
    aspect='auto'
)

fig_heatmap.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white'
)
fig_heatmap.show()

### [인사이트]
- ESG 우수 등급(A+, A) 그룹의 시가총액 중앙값(17,255억 원)이 일반/하위 등급(B~D) 그룹의 중앙값(2,300억 원)보다 약 7.5배 높은 수준을 형성한다.

-> 대형 상장주일수록 ESG 등급 관리 수준 및 성숙도가 높게 나타난다.

-> 기각 여부 애매

----
## 5. 상장기업 대상 등급 변동 현황
- [기준 데이터셋] KCGS_평가등급_2021-2025_Long.csv

##### 5-1. 최근 5개년 ESG 등급 조정 기업 수 (2021년 vs 2025년)
총 비교 대상 기업: 825개사

- 하향 기업: 389개사 (47.15%) — KCGS 평가 기준 강화 등에 따라 등급이 감소한 기업 비중이 가장 높음
- 유지 기업: 291개사 (35.27%)
- 상향 기업: 145개사 (17.58%)

In [130]:
grade_map = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}
df['ESG_점수'] = df['ESG등급'].map(grade_map)

# 중복 제거 및 피벗 (기업코드 x 평가년도)
df_unique = df.drop_duplicates(subset=['기업코드', '평가년도'])
df_pivot = df_unique.pivot(index='기업코드', columns='평가년도', values='ESG_점수')

# 2. 연도별(YoY) 등급 변동 집계
yearly_changes = []
years = [2021, 2022, 2023, 2024, 2025]

for i in range(len(years)-1):
    y1, y2 = years[i], years[i+1]
    diff = df_pivot[y2] - df_pivot[y1]
    
    up = (diff > 0).sum()
    same = (diff == 0).sum()
    down = (diff < 0).sum()
    
    yearly_changes.append({'기간': f'{y1}→{y2}', '변화구분': '상향', '기업수': up})
    yearly_changes.append({'기간': f'{y1}→{y2}', '변화구분': '유지', '기업수': same})
    yearly_changes.append({'기간': f'{y1}→{y2}', '변화구분': '하향', '기업수': down})

df_yearly_melted = pd.DataFrame(yearly_changes)

# [1] 연도별 전년 대비 ESG 등급 변동 추이 (누적 막대 차트)
fig1 = px.bar(
    df_yearly_melted,
    x='기간',
    y='기업수',
    color='변화구분',
    text='기업수',
    title='<b>[1] 연도별 전년 대비 ESG 등급 변동 추이 (2021→2025)</b>',
    labels={'기간': '평가 기간', '기업수': '기업 수 (개)', '변화구분': '등급 변동'},
    color_discrete_map={'상향': '#457B9D', '유지': '#C5D3E8', '하향': '#E07A5F'},
    category_orders={'변화구분': ['상향', '유지', '하향']}
)

fig1.update_traces(textposition='inside', texttemplate='%{y}개')
fig1.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    barmode='stack'
)
fig1.show()

-> 2024 -> 2025 등급 하향이 많아진 이유

##### 5-2. 주요 산업군별 최근 3개년 ESG 등급 추세 현황

In [131]:
grade_map = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}
df['ESG_점수'] = df['ESG등급'].map(grade_map)

df_3y = df[df['평가년도'].isin([2023, 2024, 2025])].copy()
df_3y['업종명'] = df_3y['업종명'].fillna('미분류(없음)')

df_3y_unique = df_3y.drop_duplicates(subset=['기업코드', '평가년도'])
df_pivot = df_3y_unique.pivot(index=['기업코드', '업종명'], columns='평가년도', values='ESG_점수').reset_index()

# 3개년 연속 데이터 존재하는 기업 대상
df_pivot_valid = df_pivot.dropna(subset=[2023, 2024, 2025]).copy()

# 연도별 변동량 계산
df_pivot_valid['변동_23_24'] = df_pivot_valid[2024] - df_pivot_valid[2023]
df_pivot_valid['변동_24_25'] = df_pivot_valid[2025] - df_pivot_valid[2024]

def classify_trend(row):
    d1, d2 = row['변동_23_24'], row['변동_24_25']
    if d1 >= 0 and d2 >= 0 and (d1 > 0 or d2 > 0):
        return '지속 개선(상승)'
    elif d1 <= 0 and d2 <= 0 and (d1 < 0 or d2 < 0):
        return '지속 악화(하강)'
    elif d1 == 0 and d2 == 0:
        return '등급 유지'
    else:
        return '변동/혼조'

df_pivot_valid['3개년추세'] = df_pivot_valid.apply(classify_trend, axis=1)

# 기업 수 상위 10개 산업군 필터링
top10_sectors = df_pivot_valid['업종명'].value_counts().head(10).index.tolist()
df_top10_trend = df_pivot_valid[df_pivot_valid['업종명'].isin(top10_sectors)].copy()

sector_trend_ct = pd.crosstab(df_top10_trend['업종명'], df_top10_trend['3개년추세']).reset_index()

# 시각화를 위한 Melt
sector_trend_melted = pd.melt(
    sector_trend_ct,
    id_vars=['업종명'],
    value_vars=['지속 개선(상승)', '등급 유지', '지속 악화(하강)', '변동/혼조'],
    var_name='3개년추세',
    value_name='기업수'
)

# [1] 주요 산업별 최근 3개년 ESG 등급 추세 분포 (누적 막대)
fig = px.bar(
    sector_trend_melted,
    x='업종명',
    y='기업수',
    color='3개년추세',
    title='<b>[1] TOP 10 주요 산업별 최근 3개년(2023~2025) ESG 등급 추세 분포</b>',
    labels={'업종명': '주요 산업군', '기업수': '기업 수 (개)', '3개년추세': '3개년 등급 추세'},
    color_discrete_map={
        '지속 개선(상승)': '#457B9D',  # 파스텔 딥 블루
        '등급 유지': '#C5D3E8',      # 페일 블루
        '지속 악화(하강)': '#E07A5F',  # 테라코타 오렌지
        '변동/혼조': '#A8DADC'       # 연한 민트 블루
    },
    category_orders={
        '업종명': top10_sectors,
        '3개년추세': ['지속 개선(상승)', '등급 유지', '지속 악화(하강)', '변동/혼조']
    }
)

fig.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    barmode='stack'
)
fig.show()

### [인사이트]
- 지속 개선 우수 산업

-> IT 서비스(30.2%), 전기·전자(26.7%), 제약(25.4%) 산업군은 글로벌 공급망 규제 및 ESG 공시 의무화에 선제적으로 대응하며

3년간 등급이 지속 상승하는 경향이 가장 높게 나타났습니다.


- 지속 악화 고위험 산업

-> 금속(31.7%), 기타금융(29.6%), 화학(21.1%) 산업군은 탄소배출 부담 및 내부통제 규제 강화 등의 영향을 받아

등급이 지속적으로 하락하는 기업의 비율이 높았습니다.


----
## 6. 산업별 E/S/G 강점 분석
- [기준 데이터셋] KCGS_평가등급_2021-2025_Long.csv

[ 가설 ] 화학 산업은 환경 등급이 높을 것이다. 금융 산업은 지배구조 등급이 높을 것이다.

In [132]:
# 등급 수치화 매핑 (A+=6 ~ D=1)
grade_map = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}

# 최근 연도(2025년) 또는 전체 데이터 대상 설정
df_latest = df[df['평가년도'] == 2025].copy()
df_latest['업종명'] = df_latest['업종명'].fillna('미분류(없음)')

# E, S, G 각 영역별 점수 변환
df_latest['E_점수'] = df_latest['환경'].map(grade_map)
df_latest['S_점수'] = df_latest['사회'].map(grade_map)
df_latest['G_점수'] = df_latest['지배구조'].map(grade_map)

# 2. 기업 수 상위 주요 12개 산업군 추출
top_sectors = df_latest['업종명'].value_counts().head(12).index.tolist()
df_top = df_latest[df_latest['업종명'].isin(top_sectors)]

# 3. 산업별 E, S, G 평균 점수 산출
esg_heatmap_df = df_top.groupby('업종명')[['E_점수', 'S_점수', 'G_점수']].mean()
esg_heatmap_df.columns = ['환경 (E)', '사회 (S)', '지배구조 (G)']

# 종합점수 기준 정렬 후 전치(.T)하여 행/열 전환 (산업군이 X축으로 이동)
esg_heatmap_df['종합평균'] = esg_heatmap_df.mean(axis=1)
esg_heatmap_df = esg_heatmap_df.sort_values(by='종합평균', ascending=False).drop(columns=['종합평균'])
esg_heatmap_transposed = esg_heatmap_df.T

# 4. Plotly 히트맵 작성 (행/열 전환 적용)
fig = px.imshow(
    esg_heatmap_transposed,
    labels=dict(x="주요 산업군", y="ESG 세부 영역", color="평균 점수"),
    x=esg_heatmap_transposed.columns,
    y=esg_heatmap_transposed.index,
    color_continuous_scale='Blues', # 파란색 계열 열지도
    text_auto='.2f',               # 소수점 2자리 수치 표시
    title='<b>주요 산업군별 ESG 세부 영역 평균 점수 비교</b>'
)

fig.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    xaxis_title="주요 산업군",
    yaxis_title="ESG 영역",
    coloraxis_colorbar=dict(title="평균 점수<br>(6=A+, 1=D)")
)

fig.show()

In [133]:
target_sectors = ['화학', '기타금융', '증권', '전기·전자']
df_target = df_latest[df_latest['업종명'].isin(target_sectors)].copy()

# E, S, G 중 비교하고 싶은 영역 선택 (예: 지배구조)
fig2 = px.histogram(
    df_target,
    x='업종명',
    color='지배구조',
    barnorm='percent', # 100% 누적 막대로 변환
    category_orders={'지배구조': ['A+', 'A', 'B+', 'B', 'C', 'D']},
    color_discrete_sequence=px.colors.sequential.Blues_r,
    title='<b>주요 산업별 지배구조(G) 등급 분포 비교 (100% 누적)</b>',
    labels={'업종명': '산업군', 'count': '비율 (%)'}
)

fig2.update_layout(font_family='Apple SD Gothic Neo', template='plotly_white')
fig2.show()

In [134]:
import plotly.graph_objects as go

# 비교할 주요 산업선정 (예: 화학 vs 기타금융)
compare_sectors = ['화학', '기타금융', '전기·전자']

fig3 = go.Figure()

for sector in compare_sectors:
    sector_data = esg_heatmap_df.loc[sector] # 기존 산출한 평균점수 활용
    fig3.add_trace(go.Scatterpolar(
        r=[sector_data['환경 (E)'], sector_data['사회 (S)'], sector_data['지배구조 (G)']],
        theta=['환경 (E)', '사회 (S)', '지배구조 (G)'],
        fill='toself',
        name=sector
    ))

fig3.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[1, 6])),
    title='<b>주요 산업별 ESG 영역별 상대적 강점 비교 (레이더 차트)</b>',
    font_family='Apple SD Gothic Neo',
    template='plotly_white'
)
fig3.show()

In [135]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. 최근 연도(2025년) 기준 E(환경) 우수 기업(A+, A) 필터링
df_latest = df[df['평가년도'] == 2025].copy()
df_latest['업종명'] = df_latest['업종명'].fillna('미분류(없음)')

# 컬럼명 유연 대응 ('환경' 또는 '환경등급')
e_col = '환경' if '환경' in df_latest.columns else '환경등급'

# E 우수 기업(A+, A) 추출
df_e_high = df_latest[df_latest[e_col].isin(['A+', 'A'])].copy()

# 2. 산업별 집계 (절대 기업 수 & 비중 %)
# (1) 산업별 E 우수 기업 수
sector_e_high = df_e_high['업종명'].value_counts().reset_index()
sector_e_high.columns = ['업종명', 'E_우수기업수']

# (2) 산업별 전체 평가 기업 수
sector_total = df_latest['업종명'].value_counts().reset_index()
sector_total.columns = ['업종명', '전체기업수']

# (3) 데이터 병합 및 비율 계산
sector_df = pd.merge(sector_total, sector_e_high, on='업종명', how='left').fillna(0)
sector_df['E_우수비중(%)'] = (sector_df['E_우수기업수'] / sector_df['전체기업수'] * 100).round(1)

# 최소 모수 필터링 (전체 기업 수 5개 이상인 산업만 대상)
sector_df_filtered = sector_df[sector_df['전체기업수'] >= 5].copy()

# --- [시각화 1] E 우수 기업 수 상위 TOP 10 산업 (절대 규모) ---
top10_count = sector_df_filtered.sort_values(by='E_우수기업수', ascending=False).head(10)

fig1 = px.bar(
    top10_count,
    x='E_우수기업수',
    y='업종명',
    orientation='h',
    text='E_우수기업수',
    title='<b>[1] E(환경) 우수 기업(A/A+) 보유 수 TOP 10 산업군</b>',
    labels={'E_우수기업수': '우수 기업 수 (개)', '업종명': '주요 산업군'},
    color='E_우수기업수',
    color_continuous_scale='Blues'
)
fig1.update_traces(textposition='outside')
fig1.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    yaxis=dict(autorange="reversed"),
    showlegend=False
)
fig1.show()

# --- [시각화 2] 산업 내 E 우수 기업 비중(%) TOP 10 산업 (집중도) ---
top10_ratio = sector_df_filtered.sort_values(by='E_우수비중(%)', ascending=False).head(10)

fig2 = px.bar(
    top10_ratio,
    x='E_우수비중(%)',
    y='업종명',
    orientation='h',
    text='E_우수비중(%)',
    title='<b>[2] 산업 내 E(환경) 우수 기업 비중(%) TOP 10 산업군</b>',
    labels={'E_우수비중(%)': '우수 기업 비중 (%)', '업종명': '주요 산업군'},
    color='E_우수비중(%)',
    color_continuous_scale='Blues'
)
fig2.update_traces(texttemplate='%{x}%', textposition='outside')
fig2.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    yaxis=dict(autorange="reversed"),
    showlegend=False
)
fig2.show()

### 업종별 ESG 편차 분석(양극화 확인용)

In [136]:
# 1. 데이터 정제 및 등급 수치화 (6점~1점)
df_latest = df[df['평가년도'] == 2025].copy()
df_latest['업종명'] = df_latest['업종명'].fillna('미분류(없음)')

grade_map = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}
df_latest['ESG_점수'] = df_latest['ESG등급'].map(grade_map)

# 2. 최소 10개 이상 기업이 속한 업종 대상 표준편차/평균 집계
sector_stats = df_latest.groupby('업종명')['ESG_점수'].agg(
    기업수='count',
    평균점수='mean',
    표준편차='std'
).reset_index()

# 최소 모수 필터링 (기업 수 10개 이상)
sector_stats_filtered = sector_stats[sector_stats['기업수'] >= 10].copy()

# --- [1] ESG 등급 편차가 큰 TOP 10 업종 (양극화 심화) ---
top_std_high = sector_stats_filtered.sort_values(by='표준편차', ascending=False).head(10)

fig_high = px.bar(
    top_std_high,
    x='표준편차',
    y='업종명',
    orientation='h',
    text='표준편차',
    title='<b>[1] 업종 내 ESG 등급 편차(양극화)가 큰 상위 10개 업종</b>',
    labels={'표준편차': 'ESG 점수 표준편차', '업종명': '주요 업종명'},
    color='표준편차',
    color_continuous_scale='Reds'
)
fig_high.update_traces(texttemplate='%{x:.2f}', textposition='outside')
fig_high.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    yaxis=dict(autorange="reversed"),
    showlegend=False
)
fig_high.show()

# --- [2] ESG 등급 편차가 작은 TOP 10 업종 (평준화) ---
top_std_low = sector_stats_filtered.sort_values(by='표준편차', ascending=True).head(10)

fig_low = px.bar(
    top_std_low,
    x='표준편차',
    y='업종명',
    orientation='h',
    text='표준편차',
    title='<b>[2] 업종 내 ESG 등급 편차가 작고 균일한 상위 10개 업종</b>',
    labels={'표준편차': 'ESG 점수 표준편차', '업종명': '주요 업종명'},
    color='표준편차',
    color_continuous_scale='Blues_r'
)
fig_low.update_traces(texttemplate='%{x:.2f}', textposition='outside')
fig_low.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    yaxis=dict(autorange="reversed"),
    showlegend=False
)
fig_low.show()

### 시가총액 규모(대형주/중형주/소형주)별 ESG 등급 분포

In [137]:
# 1. 데이터 준비 (2025년 기준)
df_latest = df[df['평가년도'] == 2025].copy()

# 등급 수치화 매핑 (A+=6 ~ D=1)
grade_map = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}
df_latest['ESG_점수'] = df_latest['ESG등급'].map(grade_map)

# 2. 시가총액 기준 기업 규모 그룹 분류 (Quantile 기준)
# ※ '시가총액' 컬럼이 존재하는 경우 활용
if '시가총액' in df_latest.columns:
    df_latest['시총_그룹'] = pd.qcut(
        df_latest['시가총액'],
        q=[0, 0.5, 0.8, 1.0],
        labels=['소형주 (하위 50%)', '중형주 (중위 30%)', '대형주 (상위 20%)']
    )

    # 그룹별 ESG 평균 점수 산출
    cap_summary = df_latest.groupby('시총_그룹')['ESG_점수'].mean().reset_index()

    # 3. 시가총액 그룹별 ESG 평균 점수 시각화
    fig = px.bar(
        cap_summary,
        x='시총_그룹',
        y='ESG_점수',
        text='ESG_점수',
        title='<b>[1] 시가총액 규모별 ESG 평균 점수 비교 (6점 만점)</b>',
        labels={'시총_그룹': '시가총액 규모', 'ESG_점수': 'ESG 평균 점수'},
        color='시총_그룹',
        color_discrete_sequence=['#C5D3E8', '#A8DADC', '#457B9D']
    )
    fig.update_traces(texttemplate='%{y:.2f}점', textposition='outside')
    fig.update_layout(
        font_family='Apple SD Gothic Neo',
        template='plotly_white',
        showlegend=False,
        yaxis=dict(range=[1, 6])
    )
    fig.show()

    # 4. 시가총액 그룹별 A등급 이상 우수 기업 비중(%) 산출
    df_latest['A이상_여부'] = df_latest['ESG등급'].isin(['A+', 'A']).astype(int)
    high_grade_ratio = df_latest.groupby('시총_그룹')['A이상_여부'].mean().reset_index()
    high_grade_ratio['비중(%)'] = high_grade_ratio['A이상_여부'] * 100

    fig2 = px.bar(
        high_grade_ratio,
        x='시총_그룹',
        y='비중(%)',
        text='비중(%)',
        title='<b>[2] 시가총액 규모별 A등급 이상 우수 기업 비중 (%)</b>',
        labels={'시총_그룹': '시가총액 규모', '비중(%)': 'A/A+ 등급 비중 (%)'},
        color='시총_그룹',
        color_discrete_sequence=['#C5D3E8', '#A8DADC', '#1D3557']
    )
    fig2.update_traces(texttemplate='%{y:.1f}%', textposition='outside')
    fig2.update_layout(
        font_family='Apple SD Gothic Neo',
        template='plotly_white',
        showlegend=False
    )
    fig2.show()

### ESG 성과 최고 기업(상위 3개사)과 최저 기업(하위 3개사)

In [138]:
# 1. 데이터 정제 및 등급 수치화 (A+=6 ~ D=1)
df_latest = df[df['평가년도'] == 2025].copy()
df_latest['업종명'] = df_latest['업종명'].fillna('미분류(없음)')

grade_map = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}

# 영역별 점수 컬럼 생성
df_latest['ESG_점수'] = df_latest['ESG등급'].map(grade_map)
df_latest['E_점수'] = df_latest['환경등급'].map(grade_map) if '환경등급' in df_latest else df_latest['환경'].map(grade_map)
df_latest['S_점수'] = df_latest['사회등급'].map(grade_map) if '사회등급' in df_latest else df_latest['사회'].map(grade_map)
df_latest['G_점수'] = df_latest['지배구조등급'].map(grade_map) if '지배구조등급' in df_latest else df_latest['지배구조'].map(grade_map)

# 세부 점수 합산 (동점자 처리용)
df_latest['ESG_세부합계'] = df_latest['E_점수'] + df_latest['S_점수'] + df_latest['G_점수']

# 2. 기업 수가 많은 주요 10개 업종 추출
top10_sectors = df_latest['업종명'].value_counts().head(10).index.tolist()

# 3. 업종별 상위 3개사 / 하위 3개사 추출
results = []

for sector in top10_sectors:
    sector_df = df_latest[df_latest['업종명'] == sector].copy()
    
    # 종합등급 점수 및 세부합계 기준 정렬
    sector_sorted = sector_df.sort_values(
        by=['ESG_점수', 'ESG_세부합계'], 
        ascending=[False, False]
    )
    
    # 리더 3개사 및 최하위 3개사 추출
    top3 = sector_sorted.head(3)
    bottom3 = sector_sorted.tail(3).iloc[::-1]  # 가장 낮은 기업부터 표시
    
    top3_names = ", ".join([f"{row['기업명']}({row['ESG등급']})" for _, row in top3.iterrows()])
    bottom3_names = ", ".join([f"{row['기업명']}({row['ESG등급']})" for _, row in bottom3.iterrows()])
    
    results.append({
        '주요 업종명': sector,
        '전체 기업 수': len(sector_df),
        'ESG 리더 3개사 (종합등급)': top3_names,
        'ESG 최하위 3개사 (종합등급)': bottom3_names
    })

# 결과 데이터프레임 생성
leader_vs_bottom_df = pd.DataFrame(results)
print(leader_vs_bottom_df.to_string(index=False))

 주요 업종명  전체 기업 수                           ESG 리더 3개사 (종합등급)                     ESG 최하위 3개사 (종합등급)
     화학      125               SK케미칼(A+), 현대바이오랜드(A+), DL(A)            현대바이오(D), 태경케미컬(D), 진양화학(D)
  전기·전자      101          HD현대일렉트릭(A+), LG이노텍(A), 엘에스일렉트릭(A)              주연테크(D), 에스피지(D), 아남전자(D)
     유통       74             BGF리테일(A+), GS리테일(A+), SK가스(A+)                 혜인(D), 한창(D), 플레이그램(D)
미분류(없음)       73          HD현대미포(A), 한일현대시멘트(B+), 한솔피엔에스(B+) 흥국생명보험(등급없음), 현대커머셜(등급없음), 현대캐피탈(등급없음)
   기타금융       70              KB금융(A+), 신한지주(A+), BNK금융지주(A)      한국토지신탁(D), 한국전자홀딩스(D), 에이플러스에셋(D)
     제약       69             HK이노엔(A), 동아에스티(A), 삼성바이오로직스(A)              펩트론(D), 파미셀(D), 진원생명과학(D)
운송장비·부품       64                 현대로템(A+), 현대위아(A+), HL만도(A)             태원물산(D), DH오토넥스(D), 체시스(D)
     금속       63 POSCO홀딩스(A+), LIG디펜스앤에어로스페이스(A), SK오션플랜트(A)              한일철강(D), 부국철강(D), 만호제강(D)
  기계·장비       61      현대엘리베이터(A+), HD현대건설기계(A), HD현대인프라코어(A)             화천기계(D), 한미반도체(D), 한국주강(D)


In [139]:
pip uninstall -y kaleido

Note: you may need to restart the kernel to use updated packages.


In [172]:
import plotly.express as px
import plotly.io as pio


# 비교할 업종 선택 (예: 화학)
target_sector = '전기·전자'
sector_data = df_latest[df_latest['업종명'] == target_sector].sort_values(
    by=['ESG_점수', 'ESG_세부합계'], 
    ascending=False
)

top3 = sector_data.head(3).copy()
top3['그룹'] = '리더 TOP 3'

bottom3 = sector_data.tail(3).copy()
bottom3['그룹'] = '최하위 BOTTOM 3'

compare_df = pd.concat([top3, bottom3])

# 데이터 Melt (E/S/G 세부 비교용)
compare_melted = pd.melt(
    compare_df,
    id_vars=['기업명', '그룹', 'ESG등급'],
    value_vars=['E_점수', 'S_점수', 'G_점수'],
    var_name='ESG영역',
    value_name='점수'
)
compare_melted['ESG영역'] = compare_melted['ESG영역'].replace({'E_점수': '환경(E)', 'S_점수': '사회(S)', 'G_점수': '지배구조(G)'})

# 시각화
fig = px.bar(
    compare_melted,
    x='기업명',
    y='점수',
    color='ESG영역',
    facet_col='그룹',
    barmode='group',
    title=f'<b>[{target_sector} 업종] ESG 리더 3개사 vs 최하위 3개사 세부 영역 비교</b>',
    labels={'점수': '등급 점수 (6=A+, 1=D)', '기업명': '기업명'},
    color_discrete_sequence=['#457B9D', '#A8DADC', '#E07A5F']
)

fig.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    yaxis=dict(range=[0, 6.5])
)
fig.show()

추가 궁금한 것
1. 경쟁사의 ESG 등급은 무엇인가?
2. ESG 개선을 위해 벤치마킹해야 할 기업은?

----
# 260525_팀 회의 후 추가 리서치
### <ESG등급의 편차가 큰 산업 상세 리서치>
대상 산업: 유통업, 건설업, 화학업, 전기/전자업

1. 산업 별 기업 수
2. 코스피/코스닥 상장 수 비율
3. 기업규모 별 (천억, 3천억, 5천억, 8천억, 1조, 2조, 3조 이상)  ESG 통합 등급 분포
4. 기업규모 별 (천억, 3천억, 5천억, 8천억, 1조, 2조, 3조 이상)  E/S/G 등급 각각의 3개년 추이 (막대그래프)
5. (벤치마킹) 산업 별 벤치마킹할 기업의 등급 분포
6. (벤치마킹) B+이상 등급의 회사가 지속가능경영보고서를 발행하는가?
7. (벤치마킹) B등급 이하의 기업의 ESG 활동 현황(지속가능경영보고서, 기업지배구조보고서 발행)
----

#### 1. 주요 4개 산업별 기업 수

In [141]:
import pandas as pd
import plotly.express as px

# 데이터 로드
df = pd.read_csv('/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv')

# 1. 2025년 기준 4개 산업 필터링
target_sectors = ['유통', '건설', '화학', '전기·전자']
df_2025 = df[(df['평가년도'] == 2025) & (df['업종명'].isin(target_sectors))].copy()

# 2. 산업별 기업 수 집계
sector_counts = df_2025['업종명'].value_counts().reset_index()
sector_counts.columns = ['업종명', '기업수']

# 3. 시각화
fig1 = px.bar(
    sector_counts,
    x='업종명',
    y='기업수',
    text='기업수',
    title='<b>[1] 주요 4개 산업별 평가 대상 기업 수 (2025년 기준)</b>',
    labels={'업종명': '주요 산업군', '기업수': '기업 수 (개)'},
    color='업종명',
    color_discrete_sequence=['#1D3557', '#457B9D', '#A8DADC', '#E63946']
)

fig1.update_traces(textposition='outside')
fig1.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    showlegend=False
)
fig1.show()

#### 2. 주요 4개 산업의 코스피 / 코스닥 상장 비중

In [142]:
# 100% 누적 막대 차트로 상장 시장 비율 시각화
fig2 = px.histogram(
    df_2025,
    x='업종명',
    color='시장구분',  # KOSPI / KOSDAQ
    barnorm='percent',  # % 비율 변환
    title='<b>[2] 주요 4개 산업별 KOSPI / KOSDAQ 상장 비율</b>',
    labels={'업종명': '주요 산업군', 'count': '비율 (%)', '시장구분': '상장 시장'},
    color_discrete_sequence=['#1D3557', '#457B9D']
)

fig2.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white'
)
fig2.show()

#### 3. 기업규모(시가총액 구간)별 ESG 통합 등급 분포 (원화 표기)

In [146]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# VS Code 전용 렌더러 설정
pio.renderers.default = "vscode"

# 1. 데이터 로드 및 2025년 4개 대상 산업 필터링
df = pd.read_csv('/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv')
target_sectors = ['유통', '건설', '화학', '전기·전자']
df_2025 = df[(df['평가년도'] == 2025) & (df['업종명'].isin(target_sectors))].copy()

# 2. 시가총액 구간 범주화 (원화 기준)
bins = [0, 1e11, 3e11, 5e11, 8e11, 1e12, 2e12, 3e12, 5e12, 1e13, np.inf]
labels = [
    '1천억 미만', '1천억 ~ 3천억', '3천억 ~ 5천억', '5천억 ~ 8천억',
    '8천억 ~ 1조', '1조 ~ 2조', '2조 ~ 3조', '3조 ~ 5조', '5조 ~ 10조', '10조 이상'
]

df_cap = df_2025.dropna(subset=['시가총액']).copy()
df_cap['시총_구간'] = pd.cut(df_cap['시가총액'], bins=bins, labels=labels)

# 3. Y축=ESG등급, X축=시가총액 구간으로 교차표 생성 (열 기준 비중 %)
crosstab_esg = pd.crosstab(
    df_cap['ESG등급'], 
    df_cap['시총_구간'], 
    normalize='columns'
) * 100

# ESG 등급 y축 정렬 (A+ ~ D)
grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']
existing_grades = [g for g in grade_order if g in crosstab_esg.index]
crosstab_esg = crosstab_esg.loc[existing_grades]

# 4. 히트맵 시각화
fig3 = px.imshow(
    crosstab_esg,
    labels=dict(x="시가총액 구간", y="ESG 통합 등급", color="비중 (%)"),
    x=crosstab_esg.columns,
    y=crosstab_esg.index,
    color_continuous_scale='Blues',
    text_auto='.1f',
    title='<b>[3] 시가총액 규모별 ESG 통합 등급 분포 (y축: ESG등급)</b>'
)

fig3.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white'
)

fig3.show()

#### 4. 기업규모(시가총액)별 E / S / G 세부 등급 3개년 평균 추이 (2023 ~ 2025년)

In [160]:
# 1. 최근 3개년(2023~2025년) 4개 산업 데이터 추출
df_3y = df[(df['평가년도'].isin([2023, 2024, 2025])) & (df['업종명'].isin(target_sectors))].copy()
df_3y = df_3y.dropna(subset=['시가총액']).copy()

# 등급 수치화 매핑 (6점~1점)
grade_map = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}
df_3y['E_점수'] = df_3y['환경'].map(grade_map)
df_3y['S_점수'] = df_3y['사회'].map(grade_map)
df_3y['G_점수'] = df_3y['지배구조'].map(grade_map)

# 시가총액 구간 반영
df_3y['시총_구간'] = pd.cut(df_3y['시가총액'], bins=bins, labels=labels)

# 2. 연도/시총구간별 E, S, G 평균 산출
trend_df = df_3y.groupby(['평가년도', '시총_구간'], observed=False)[['E_점수', 'S_점수', 'G_점수']].mean().reset_index()

# Melt 형태로 변환
trend_melted = pd.melt(
    trend_df,
    id_vars=['평가년도', '시총_구간'],
    value_vars=['E_점수', 'S_점수', 'G_점수'],
    var_name='ESG영역',
    value_name='평균점수'
)
trend_melted['영역'] = trend_melted['ESG영역'].replace({'E': 'E', 'S_점수': 'S', 'G_점수': 'G'})
trend_melted['평가년도'] = trend_melted['평가년도'].astype(str)

# 3. 영역별 3개년 추이 막대 그래프 시각화
fig4 = px.bar(
    trend_melted,
    x='시총_구간',
    y='평균점수',
    color='평가년도',
    facet_row='ESG영역',
    barmode='group',
    title='<b>[4] 기업 규모(시가총액)별 E/S/G 세부 영역 3개년 평균 점수 추이</b>',
    labels={'시총_구간': '시가총액 구간', '평균점수': '평균 점수', '평가년도': '연도'},
    color_discrete_sequence=['#A8DADC', '#457B9D', '#1D3557']
)

fig4.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    yaxis=dict(range=[1, 6])
)
fig4.show()

#### 5. 주요 4개 산업별 벤치마킹 기업 등급 분포 (2025년 기준)

In [161]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# VS Code 주피터 전용 렌더러 설정 (kaleido 라이브러리 없이 실행 가능)
pio.renderers.default = "vscode"

# 1. 데이터 로드 및 2025년 대상 4개 산업 필터링
file_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv'
df = pd.read_csv(file_path)

target_sectors = ['유통', '건설', '화학', '전기·전자']
df_2025 = df[(df['평가년도'] == 2025) & (df['업종명'].isin(target_sectors))].copy()

# 2. 등급 정렬 순서 정의 (A+ ~ D)
grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']

# 3. 산업별 / ESG 등급별 기업 수 집계
grade_counts = df_2025.groupby(['업종명', 'ESG등급']).size().reset_index(name='기업수')

# 4. 차트 생성 (그룹 막대 그래프)
fig = px.bar(
    grade_counts,
    x='업종명',
    y='기업수',
    color='ESG등급',
    barmode='group',
    category_orders={'ESG등급': grade_order, '업종명': target_sectors},
    title='<b>[EDA] 주요 4개 산업별 벤치마킹 ESG 등급 분포 (2025년 기준)</b>',
    labels={'업종명': '주요 산업군', '기업수': '기업 수 (개)', 'ESG등급': 'ESG 통합 등급'},
    # 우수 등급(A+, A)을 강조하고 하위 등급으로 갈수록 차분해지는 파스텔 톤 팔레트
    color_discrete_map={
        'A+': '#1D3557',  # 딥 네이비 (최우수 벤치마크)
        'A':  '#457B9D',  # 블루 (우수 벤치마크)
        'B+': '#A8DADC',  # 민트 블루 (양호)
        'B':  '#F4A261',  # 라이트 오렌지 (보통)
        'C':  '#E76F51',  # 웜 코랄 (미흡)
        'D':  '#E63946'   # 레드 (취약)
    }
)

# 5. 레이아웃 세부 설정
fig.update_traces(
    texttemplate='%{y}',       # 막대 상단에 기업 수 표기
    textposition='outside'     # 숫자를 막대 외부에 배치
)

fig.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    legend_title_text='ESG 통합 등급',
    font=dict(color='#4A4A4A'),
    yaxis=dict(showgrid=True, gridcolor='#F0F0F0', title='기업 수 (개)'),
    xaxis=dict(title='주요 산업군'),
    bargap=0.2,               # 산업 간 간격
    bargroupgap=0.05           # 동일 산업 내 등급 막대 간 간격
)

fig.show()

In [162]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# VS Code 전용 렌더러 설정
pio.renderers.default = "vscode"

# 1. 데이터 로드 및 2025년 주요 4개 산업 필터링
file_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv'
df = pd.read_csv(file_path)

target_sectors = ['유통', '건설', '화학', '전기·전자']
df_2025 = df[(df['평가년도'] == 2025) & (df['업종명'].isin(target_sectors))].copy()

# 2. 피벗 테이블(Cross-tab) 생성
grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']
pivot_table = pd.crosstab(df_2025['업종명'], df_2025['ESG등급'])

# 등급 컬럼 순서 정렬
for g in grade_order:
    if g not in pivot_table.columns:
        pivot_table[g] = 0
pivot_table = pivot_table[grade_order]

# 합계 열 추가
pivot_table['합계'] = pivot_table.sum(axis=1)

# 합계 행 추가
total_row = pivot_table.sum(axis=0)
total_row.name = '전체 합계'
pivot_table = pd.concat([pivot_table, pd.DataFrame(total_row).T])

# 인덱스(업종명)를 컬럼으로 리셋
table_df = pivot_table.reset_index()
table_df.rename(columns={'index': '업종명'}, inplace=True)

# 3. Plotly go.Table 시각화 생성
fig_table = go.Figure(data=[go.Table(
    # 헤더(Header) 설정
    header=dict(
        values=[f'<b>{col}</b>' for col in table_df.columns],
        fill_color='#1D3557',       # 헤더 배경색 (딥 네이비)
        align=['center'] * len(table_df.columns),
        font=dict(color='white', size=14, family='Apple SD Gothic Neo'),
        height=40
    ),
    # 데이터 셀(Cells) 설정
    cells=dict(
        values=[table_df[col] for col in table_df.columns],
        fill_color=[
            # '전체 합계' 행과 일반 행의 배경색 구분
            ['#F8F9FA' if val != '전체 합계' else '#E9ECEF' for val in table_df['업종명']]
        ],
        align=['center'] * len(table_df.columns),
        font=dict(color='#212529', size=13, family='Apple SD Gothic Neo'),
        height=32
    )
)])

# 4. 레이아웃 세부 조정
fig_table.update_layout(
    title='<b>[표 시각화] 주요 4개 산업별 ESG 통합 등급 분포 (2025년 기준)</b>',
    title_font_size=16,
    width=850,
    height=350,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig_table.show()

In [166]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import zipfile
import io
import time

# ----------------------------------------------------
# 0. 설정
# ----------------------------------------------------
DART_API_KEY = "4c056faa738bcb25aa545b05a5b1d69137a71f39"
YEAR = "2025"  # 분석 대상 연도 (필요에 따라 변경)
FILE_PATH = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv'

# 데이터 로드 (주요 4개 산업)
df = pd.read_csv(FILE_PATH)
target_sectors = ['유통', '건설', '화학', '전기·전자']
df_target = df[(df['평가년도'] == int(YEAR)) & (df['업종명'].isin(target_sectors))].copy()

# 기업코드 6자리 문자열로 정규화
df_target['stock_code'] = df_target['기업코드'].astype(str).str.zfill(6)
unique_stocks = df_target[['stock_code', '기업명']].drop_duplicates()

# ----------------------------------------------------
# 1. DART corp_code(고유번호) 맵핑 파일 다운로드
# ----------------------------------------------------
print("1. DART 기업 고유번호 파일 다운로드 중...")
corp_code_url = f"https://opendart.fss.or.kr/api/corpCode.xml?crtfc_key={DART_API_KEY}"
res = requests.get(corp_code_url)

corp_map = {}
if res.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(res.content)) as z:
        xml_data = z.read('CORPCODE.xml')
        root = ET.fromstring(xml_data)
        for list_tag in root.findall('list'):
            stock_code = list_tag.findtext('stock_code').strip()
            corp_code = list_tag.findtext('corp_code').strip()
            if stock_code:  # 상장사만 매핑
                corp_map[stock_code] = corp_code

# 데이터프레임에 DART corp_code 매핑
unique_stocks['corp_code'] = unique_stocks['stock_code'].map(corp_map)
print(f"매핑 완료: 전체 {len(unique_stocks)}개 기업 중 {unique_stocks['corp_code'].notnull().sum()}개 매핑됨")

# ----------------------------------------------------
# 2. OpenDART 공시검색 API 호출 (이진분류)
# ----------------------------------------------------
print("2. DART 공시 조회 시작...")

sustainability_status = {}  # 지속가능경영보고서 (1/0)
governance_status = {}     # 기업지배구조보고서 (1/0)

bgn_de = f"{YEAR}0101"
end_de = f"{YEAR}1231"

for idx, row in unique_stocks.iterrows():
    corp_c = row['corp_code']
    stock_c = row['stock_code']
    
    if pd.isna(corp_c):
        sustainability_status[stock_c] = 0
        governance_status[stock_c] = 0
        continue
    
    # DART 공시검색 API
    list_url = f"https://opendart.fss.or.kr/api/list.json?crtfc_key={DART_API_KEY}&corp_code={corp_c}&bgn_de={bgn_de}&end_de={end_de}&page_count=100"
    resp = requests.get(list_url).json()
    
    sus_flag = 0
    gov_flag = 0
    
    if resp.get('status') == '000' and 'list' in resp:
        for item in resp['list']:
            report_nm = item.get('report_nm', '')
            
            # 지속가능경영보고서 (자율공시 / 지속가능경영 / ESG보고서 키워드 검색)
            if any(k in report_nm for k in ['지속가능경영', '지속가능', 'ESG보고서', 'ESG 보고서']):
                sus_flag = 1
                
            # 기업지배구조보고서 (지배구조 보고서 키워드 검색)
            if '지배구조' in report_nm and '보고서' in report_nm:
                gov_flag = 1
                
    sustainability_status[stock_c] = sus_flag
    governance_status[stock_c] = gov_flag
    
    time.sleep(0.1)  # API 호출 제한 방지

# 메인 데이터프레임에 결과 결합
df_target['지속가능경영보고서_발행'] = df_target['stock_code'].map(sustainability_status)
df_target['기업지배구조보고서_발행'] = df_target['stock_code'].map(governance_status)

print("\n=== 공시 수집 및 이진분류 완료 ===")
print(df_target[['기업명', 'ESG등급', '지속가능경영보고서_발행', '기업지배구조보고서_발행']].head(10))

1. DART 기업 고유번호 파일 다운로드 중...
매핑 완료: 전체 329개 기업 중 329개 매핑됨
2. DART 공시 조회 시작...

=== 공시 수집 및 이진분류 완료 ===
            기업명 ESG등급  지속가능경영보고서_발행  기업지배구조보고서_발행
17       BGF리테일    A+             1             1
22   BGF에코머티리얼즈     B             0             0
73      CJ프레시웨이     A             0             0
105       DB하이텍    B+             1             1
120          DL     A             0             1
128       DL이앤씨     A             1             1
132     DN오토모티브     C             0             1
135       DRB동일    B+             0             1
150        DS단석    B+             1             1
152          E1    B+             1             1


In [167]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# VS Code 렌더러 설정
pio.renderers.default = "vscode"

grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']

# ----------------------------------------------------
# [EDA 1] 지속가능경영보고서 등급별/산업별 공시 비율
# ----------------------------------------------------
sus_summary = df_target.groupby(['업종명', 'ESG등급'], observed=False)['지속가능경영보고서_발행'].mean().reset_index()
sus_summary['발행비율(%)'] = (sus_summary['지속가능경영보고서_발행'] * 100).round(1)

fig_sus = px.bar(
    sus_summary,
    x='ESG등급',
    y='발행비율(%)',
    color='업종명',
    barmode='group',
    category_orders={'ESG등급': grade_order, '업종명': target_sectors},
    title='<b>[1] 주요 4개 산업의 ESG 등급별 지속가능경영보고서 공시 비율 (%)</b>',
    labels={'발행비율(%)': '공시 비율 (%)', 'ESG등급': 'ESG 통합 등급'},
    color_discrete_sequence=['#1D3557', '#457B9D', '#A8DADC', '#E63946']
)

fig_sus.update_traces(texttemplate='%{y}%', textposition='outside')
fig_sus.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    yaxis=dict(range=[0, 115])
)
fig_sus.show()

# ----------------------------------------------------
# [EDA 2] 기업지배구조보고서 시장별/등급별 공시 비율
# ----------------------------------------------------
gov_summary = df_target.groupby(['시장구분', 'ESG등급'], observed=False)['기업지배구조보고서_발행'].mean().reset_index()
gov_summary['발행비율(%)'] = (gov_summary['기업지배구조보고서_발행'] * 100).round(1)

fig_gov = px.bar(
    gov_summary,
    x='ESG등급',
    y='발행비율(%)',
    color='시장구분',  # KOSPI / KOSDAQ
    barmode='group',
    category_orders={'ESG등급': grade_order},
    title='<b>[2] 상장 시장(KOSPI / KOSDAQ) 및 등급별 기업지배구조보고서 공시 비율 (%)</b>',
    labels={'발행비율(%)': '공시 비율 (%)', 'ESG등급': 'ESG 통합 등급'},
    color_discrete_sequence=['#2A9D8F', '#E76F51']
)

fig_gov.update_traces(texttemplate='%{y}%', textposition='outside')
fig_gov.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    yaxis=dict(range=[0, 115])
)
fig_gov.show()

# ----------------------------------------------------
# [EDA 3] 2개 보고서 동시 발행 여부 교차표 (Table)
# ----------------------------------------------------
df_target['보고서_발행_유형'] = '미발행'
df_target.loc[(df_target['지속가능경영보고서_발행'] == 1) & (df_target['기업지배구조보고서_발행'] == 0), '보고서_발행_유형'] = '지속가능경영만'
df_target.loc[(df_target['지속가능경영보고서_발행'] == 0) & (df_target['기업지배구조보고서_발행'] == 1), '보고서_발행_유형'] = '지배구조만'
df_target.loc[(df_target['지속가능경영보고서_발행'] == 1) & (df_target['기업지배구조보고서_발행'] == 1), '보고서_발행_유형'] = '둘 다 발행'

cross_tab = pd.crosstab(df_target['ESG등급'], df_target['보고서_발행_유형'])
for g in grade_order:
    if g not in cross_tab.index:
        cross_tab.loc[g] = 0
cross_tab = cross_tab.loc[grade_order].reset_index()

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=[f'<b>{col}</b>' for col in cross_tab.columns],
        fill_color='#1D3557',
        align='center',
        font=dict(color='white', size=14, family='Apple SD Gothic Neo')
    ),
    cells=dict(
        values=[cross_tab[col] for col in cross_tab.columns],
        fill_color='#F8F9FA',
        align='center',
        font=dict(color='#212529', size=13, family='Apple SD Gothic Neo')
    )
)])

fig_table.update_layout(
    title='<b>[3] ESG 등급별 보고서 발행 유형 기업 수 분포</b>',
    width=750, height=300
)
fig_table.show()

In [169]:
# OpenDART 호출 결과가 반영된 df_target을 CSV 및 엑셀 파일로 저장
df_target.to_csv('KCGS_DART_ESG_보고서공시_2025.csv', index=False, encoding='utf-8-sig')
df_target.to_excel('KCGS_DART_ESG_보고서공시_2025.xlsx', index=False)

print("데이터셋 저장 완료: KCGS_DART_ESG_보고서공시_2025.csv")

import os
print("현재 파일 저장 경로:", os.getcwd())

데이터셋 저장 완료: KCGS_DART_ESG_보고서공시_2025.csv
현재 파일 저장 경로: /Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/04_파이썬


In [171]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# VS Code 또는 Jupyter 환경 렌더러 설정
pio.renderers.default = "vscode"  # 웹 브라우저 팝업으로 볼 경우 "browser"로 변경

# 이미 저장했거나 수집 완료된 df_target 사용
# df_target = pd.read_csv('KCGS_DART_ESG_보고서공시_2025.csv')

# ----------------------------------------------------
# [EDA 1] B+ 이상 우수 기업의 지속가능경영보고서 발행 비중
# ----------------------------------------------------
b_plus_df = df_target[df_target['ESG등급'].isin(['A+', 'A', 'B+'])].copy()

eda1 = b_plus_df.groupby('ESG등급', observed=False)['지속가능경영보고서_발행'].agg(
    전체기업수='count',
    발행기업수='sum',
    발행비율=lambda x: round(x.mean() * 100, 1)
).reset_index()

# 등급 정렬
eda1['ESG등급'] = pd.Categorical(eda1['ESG등급'], categories=['A+', 'A', 'B+'], ordered=True)
eda1 = eda1.sort_values('ESG등급')

print("=== [EDA 1] B+ 이상 우수 기업 집계 ===")
print(eda1)

# Plotly 시각화 [EDA 1]
fig1 = px.bar(
    eda1, 
    x='ESG등급', 
    y='발행비율',
    color='ESG등급',
    text_auto='.1f',
    title='<b>[EDA 1] B+ 이상 우수 기업 지속가능경영보고서 발행 비율 (%)</b>',
    labels={'발행비율': '발행 비율 (%)', 'ESG등급': 'ESG 통합 등급'},
    color_discrete_map={'A+': '#1D3557', 'A': '#457B9D', 'B+': '#A8DADC'}
)

fig1.update_traces(texttemplate='%{y:.1f}%', textposition='outside')
fig1.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    yaxis=dict(range=[0, 115]),
    showlegend=False
)

fig1.show()

# ----------------------------------------------------
# [EDA 2] B등급 이하 하위 기업의 ESG 보고서 2종 발행 비율 비교
# ----------------------------------------------------
b_below_df = df_target[df_target['ESG등급'].isin(['B', 'C', 'D'])].copy()

# 지속가능경영보고서 집계
eda2_sus = b_below_df.groupby('ESG등급', observed=False)['지속가능경영보고서_발행'].agg(
    전체기업수='count',
    발행기업수='sum',
    발행비율=lambda x: round(x.mean() * 100, 1)
).reset_index()
eda2_sus['보고서종류'] = '지속가능경영보고서'

# 기업지배구조보고서 집계
eda2_gov = b_below_df.groupby('ESG등급', observed=False)['기업지배구조보고서_발행'].agg(
    전체기업수='count',
    발행기업수='sum',
    발행비율=lambda x: round(x.mean() * 100, 1)
).reset_index()
eda2_gov['보고서종류'] = '기업지배구조보고서'

# 병합 및 정렬
eda2 = pd.concat([eda2_sus, eda2_gov], ignore_index=True)
eda2['ESG등급'] = pd.Categorical(eda2['ESG등급'], categories=['B', 'C', 'D'], ordered=True)
eda2 = eda2.sort_values('ESG등급')

print("\n=== [EDA 2] B등급 이하 하위 기업 집계 ===")
print(eda2)

# Plotly 시각화 [EDA 2]
fig2 = px.bar(
    eda2, 
    x='ESG등급', 
    y='발행비율',
    color='보고서종류',
    barmode='group',
    text_auto='.1f',
    title='<b>[EDA 2] B등급 이하 하위 기업 ESG 보고서 2종 발행 비율 비교 (%)</b>',
    labels={'발행비율': '발행 비율 (%)', 'ESG등급': 'ESG 통합 등급'},
    color_discrete_map={'지속가능경영보고서': '#2A9D8F', '기업지배구조보고서': '#E76F51'}
)

fig2.update_traces(texttemplate='%{y:.1f}%', textposition='outside')
fig2.update_layout(
    font_family='Apple SD Gothic Neo',
    template='plotly_white',
    yaxis=dict(range=[0, 115])
)

fig2.show()

=== [EDA 1] B+ 이상 우수 기업 집계 ===
  ESG등급  전체기업수  발행기업수  발행비율
1    A+     10      7  70.0
0     A     65     47  72.3
2    B+     46     15  32.6



=== [EDA 2] B등급 이하 하위 기업 집계 ===
  ESG등급  전체기업수  발행기업수  발행비율      보고서종류
0     B     24      9  37.5  지속가능경영보고서
3     B     24     19  79.2  기업지배구조보고서
1     C     66      1   1.5  지속가능경영보고서
4     C     66     26  39.4  기업지배구조보고서
2     D    118      0   0.0  지속가능경영보고서
5     D    118     17  14.4  기업지배구조보고서
